# 📝 Notebook Summary — 05_dbutils_write_emp_data

## What This Notebook Does

This notebook demonstrates a **parameterized data pipeline** that reads employee data from a CSV file, filters it by a department (passed via a widget), and writes the filtered results to a managed Delta table. It ends by returning the row count as the notebook exit value, which can be used by a parent job.

## Step-by-Step Breakdown

| Cell | Action | Key Function / Concept |
| --- | --- | --- |
| 1 | Preview the raw CSV file head | `dbutils.fs.head()` — read first bytes of a file |
| 2 | Create a text input widget for department name | `dbutils.widgets.text()` — parameterize notebooks |
| 3 | Retrieve the widget value into a Python variable | `dbutils.widgets.get()` — read widget input at runtime |
| 4 | Print the widget value for verification | `print()` — basic debugging |
| 5 | Read CSV into a Spark DataFrame | `spark.read.csv()` with `header=True` |
| 6 | Display first 20 rows of the DataFrame | `df.show(20)` — inspect data |
| 7 | Filter by department (case-insensitive) and active employees only | `df.where()` with `upper()` and `active_record = '1'` |
| 8 | Display the filtered DataFrame | `filtered_df.show(20)` |
| 9 | Count rows; if > 0, write to managed table; else print message | `df.write.mode('overwrite').saveAsTable()` — conditional write logic |
| 10 | Display help for notebook utilities | `dbutils.notebook.help()` — explore available functions |
| 11 | Exit the notebook and return the row count | `dbutils.notebook.exit()` — pass output to calling job |

## Key Topics Covered

### 1. 🔧 `dbutils.widgets` — Parameterization

* **`dbutils.widgets.text(name, default, label)`** creates an input text box in the notebook UI.
* **`dbutils.widgets.get(name)`** retrieves the current value at runtime.
* This allows the same notebook to be reused for different departments (e.g., `Sales`, `Marketing`, `IT`) without code changes — ideal for **job automation** where the widget value can be set as a job parameter.

### 2. 📂 `dbutils.fs` — File System Utilities

* **`dbutils.fs.head(path)`** reads the first ~1 KB of a file — useful for quickly inspecting CSV headers and sample rows without loading the entire file.

### 3. 🐍 `dbutils.notebook` — Notebook Lifecycle

* **`dbutils.notebook.exit(value)`** terminates the notebook and returns `value` to the calling job. This is critical for **multi-task jobs** where downstream tasks need the output (e.g., row count) to make branching decisions.
* The exit value can be retrieved in the parent job using `dbutils.notebook.run()` and its return value.

### 4. 📊 Spark DataFrame Operations

* **`spark.read.csv(path, header=True)`** — loads CSV data with the first row as column names.
* **`df.where(condition)`** — filters rows; uses SQL-style string expressions with `upper()` for case-insensitive matching.
* **`df.count()`** — triggers an action to count matching rows.
* **`df.show(n)`** — displays `n` rows in tabular format.

### 5. 💾 Writing to Managed Tables

* **`df.write.mode('overwrite').saveAsTable("catalog.schema.table")`** — writes the DataFrame as a managed Delta table in Unity Catalog.
* Overwrite mode replaces the table contents each run — useful for full-refresh patterns.
* The table name is **dynamically generated** using the widget value: `dev.bronze.dept_{_dept}` (e.g., `dev.bronze.dept_Sales`).

### 6. 🔀 Conditional Logic

* The `if _count > 0` check prevents writing empty tables — a guard pattern that avoids creating empty/meaningless tables and provides a clear log message instead.

## Data Schema (8 Columns)

| Column | Type | Description |
| --- | --- | --- |
| `employee_id` | string | Unique employee identifier |
| `employee_name` | string | Employee full name |
| `department` | string | Department (e.g., SALES) |
| `region` | string | Geographic region |
| `employee_key` | string | Business key |
| `active_record` | string | `1` = active, `0` = inactive |
| `active_record_start` | string | Employment start date |
| `active_record_end` | string | Employment end date (NULL if still active) |

## Important Points & Best Practices

* **Case-insensitive filtering**: Using `upper(department) == upper('{_dept}')` ensures the widget input matches regardless of casing.
* **Guard before write**: Checking `_count > 0` avoids creating empty tables.
* **Dynamic table naming**: Table name derived from the widget value makes this reusable across departments.
* **Exit value for orchestration**: `dbutils.notebook.exit(_count)` enables parent jobs to use the result for conditional branching.
* **Widget defaults**: Setting a default value (e.g., `"Sales"`) lets you test interactively before wiring it into a job.

In [0]:
# View data at location /databricks-datasets/retail-org/company_employees/company_employees.csv

dbutils.fs.head('/databricks-datasets/retail-org/company_employees/company_employees.csv')

'employee_id,employee_name,department,region,employee_key,active_record,active_record_start,active_record_end\n0,"COOK,  KIMBERLY",SALES,Northeast,48001,1,2010-10-30,\n1,"BOYLAN,  CHAD A",SALES,Northeast,48002,1,2019-03-04,\n2,"SANDOVAL JR,  DANIEL",SALES,Northeast,48003,0,2016-07-19,2017-04-19\n3,"APOSTOLOS,  CONSTANTIN E",SALES,Northeast,48004,0,2011-09-06,2012-06-06\n4,"LIPINSKI,  TINA L",SALES,Northeast,48005,1,2017-01-25,\n5,"DAHLSTROM,  SCOTT M",SALES,Northeast,48006,1,2013-10-22,\n6,"ORTIZ,  FRANCISCO",SALES,Northeast,48007,1,2013-06-09,\n7,"LOZADA,  JOSUE",SALES,Northeast,48008,1,2017-01-23,\n8,"MC GHEE,  CYNTHIA L",SALES,Northeast,48009,1,2019-10-23,\n9,"BELL,  REGINALD V",SALES,Northeast,48010,1,2015-03-31,\n10,"MORALES,  JOSE D",SALES,Northeast,48011,1,2015-11-04,\n11,"FIGUEROA,  KATHERINE A",SALES,Northeast,48012,1,2015-11-01,\n12,"BETTS,  TRAVIS L",SALES,Northeast,48013,1,2019-08-11,\n13,"PFEIFFER,  MICHAEL J",SALES,Northeast,48014,1,2013-10-30,\n14,"ARMSTRONG,  COURTNEY",

In [0]:
dbutils.widgets.text("dept","","Department Name")

In [0]:
_dept = dbutils.widgets.get("dept")

In [0]:
print(_dept)

Sales


In [0]:
df=spark.read.csv('/databricks-datasets/retail-org/company_employees/company_employees.csv',header=True)

In [0]:
df.show(20)

+-----------+--------------------+----------+---------+------------+-------------+-------------------+-----------------+
|employee_id|       employee_name|department|   region|employee_key|active_record|active_record_start|active_record_end|
+-----------+--------------------+----------+---------+------------+-------------+-------------------+-----------------+
|          0|     COOK,  KIMBERLY|     SALES|Northeast|       48001|            1|         2010-10-30|             NULL|
|          1|     BOYLAN,  CHAD A|     SALES|Northeast|       48002|            1|         2019-03-04|             NULL|
|          2|SANDOVAL JR,  DANIEL|     SALES|Northeast|       48003|            0|         2016-07-19|       2017-04-19|
|          3|APOSTOLOS,  CONST...|     SALES|Northeast|       48004|            0|         2011-09-06|       2012-06-06|
|          4|   LIPINSKI,  TINA L|     SALES|Northeast|       48005|            1|         2017-01-25|             NULL|
|          5| DAHLSTROM,  SCOTT 

In [0]:
# Filter data based on department and active_record
filtered_df = df.where(f"upper(department)==upper('{_dept}') and active_record = '1' ")

In [0]:
filtered_df.show(20)

+-----------+--------------------+----------+---------+------------+-------------+-------------------+-----------------+
|employee_id|       employee_name|department|   region|employee_key|active_record|active_record_start|active_record_end|
+-----------+--------------------+----------+---------+------------+-------------+-------------------+-----------------+
|          0|     COOK,  KIMBERLY|     SALES|Northeast|       48001|            1|         2010-10-30|             NULL|
|          1|     BOYLAN,  CHAD A|     SALES|Northeast|       48002|            1|         2019-03-04|             NULL|
|          4|   LIPINSKI,  TINA L|     SALES|Northeast|       48005|            1|         2017-01-25|             NULL|
|          5| DAHLSTROM,  SCOTT M|     SALES|Northeast|       48006|            1|         2013-10-22|             NULL|
|          6|   ORTIZ,  FRANCISCO|     SALES|Northeast|       48007|            1|         2013-06-09|             NULL|
|          7|      LOZADA,  JOSU

In [0]:
_count = filtered_df.count()
if _count > 0:
    filtered_df.write.mode('overwrite').saveAsTable(f"dev.bronze.dept_{_dept}")
    print(f"Data written to table dev.bronze.dept_{_dept}")
    display(spark.table(f"dev.bronze.dept_{_dept}"))
else:
    print(f"No data found for department {_dept}")

Data written to table dev.bronze.dept_Sales


employee_id,employee_name,department,region,employee_key,active_record,active_record_start,active_record_end
0,"COOK, KIMBERLY",SALES,Northeast,48001,1,2010-10-30,null
1,"BOYLAN, CHAD A",SALES,Northeast,48002,1,2019-03-04,null
4,"LIPINSKI, TINA L",SALES,Northeast,48005,1,2017-01-25,null
5,"DAHLSTROM, SCOTT M",SALES,Northeast,48006,1,2013-10-22,null
6,"ORTIZ, FRANCISCO",SALES,Northeast,48007,1,2013-06-09,null
7,"LOZADA, JOSUE",SALES,Northeast,48008,1,2017-01-23,null
8,"MC GHEE, CYNTHIA L",SALES,Northeast,48009,1,2019-10-23,null
9,"BELL, REGINALD V",SALES,Northeast,48010,1,2015-03-31,null
10,"MORALES, JOSE D",SALES,Northeast,48011,1,2015-11-04,null
11,"FIGUEROA, KATHERINE A",SALES,Northeast,48012,1,2015-11-01,null


In [0]:
dbutils.notebook.help()

The notebook module.
 exit(value: String): void -> This method lets you exit a notebook with a value run(path: String, timeoutSeconds: int, arguments: Map): String -> This method runs a notebook and returns its exit value

In [0]:
dbutils.notebook.exit(_count)